# Examen ICCD753 — Recuperación de Información: SCIFACT

**Dataset asignado:** `BeIR/scifact` (corpus y consultas) + `BeIR/scifact-qrels` (`test`).  
**Objetivo:** recuperar documentos completos con **TF-IDF + similitud coseno** y **BM25**,
evaluarlos con **Precision@10, Recall@10 y NDCG@10** y exponerlos como servicio web REST.

**Servicio:** `GET /health` y `POST /search`. No se desarrolla interfaz gráfica.

**Reproducibilidad:** Python 3.11, dependencias en `requirements-dev.txt`, misma lógica para notebook y API.
Ejecutar **Run All** y guardar para entregar este notebook con las salidas reales.

## 1. Preparación y carga del corpus

Se utilizan identificadores `_id` originales, sin segmentar los documentos. Se descargan
los subconjuntos `corpus`, `queries` y **solo qrels de `test`**. Los qrels no forman parte de la indexación.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "src").is_dir(), "Ejecuta el notebook dentro del proyecto SCIFACT_Examen_RI"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src.dataset import prepare_data, load_corpus, load_queries, load_test_qrels
from src.preprocess import preprocess
from src.retrieval import SciFactSearchEngine
from src.evaluation import evaluate_models

counts = prepare_data()
corpus = load_corpus()
queries = load_queries()
test_qrels = load_test_qrels()
print("Filas descargadas:", counts)
print("QIDs evaluables en test:", sum(any(s > 0 for s in rels.values()) for rels in test_qrels.values()))
print("Primer documento:", {"_id": corpus[0]["_id"], "title": corpus[0]["title"], "text_preview": corpus[0]["text"][:150]})

## 2. Auditoría y decisiones de preprocesamiento

**Preprocesamiento compartido:** NFKC → minúsculas → tokens alfanuméricos →
stopwords inglesas (se conservan las negaciones) → stemming Porter.

La colección se indexa con `title + text`. El resultado siempre corresponde al documento original.
Se comprueba integridad referencial y unicidad antes de recuperar.

In [ ]:
doc_ids = {d["_id"] for d in corpus}
query_ids = set(queries)
assert len(doc_ids) == len(corpus), "Document IDs duplicados"
assert set(test_qrels).issubset(query_ids), "Qrels con consultas faltantes"
assert all(set(rels).issubset(doc_ids) for rels in test_qrels.values()), "Qrels con documentos faltantes"
print("Documentos:", len(corpus), "| Consultas:", len(queries), "| QIDs con qrels test:", len(test_qrels))
print("Texto de prueba:", "Aspirin is NOT preventing some infections")
print("Tokens procesados:", preprocess("Aspirin is NOT preventing some infections"))

## 3. Construcción del índice invertido, TF-IDF y BM25

El **índice invertido explícito** conecta cada término con los identificadores originales.
TF-IDF usa normalización L2, de manera que el producto punto equivale a coseno.
BM25 usa `k1=1.5`, `b=0.75`. No hay embeddings ni fragmentos.

In [ ]:
engine = SciFactSearchEngine(corpus)
print("Documentos indexados:", len(engine.document_ids))
print("Vocabulario:", len(engine.vocabulary))
print("Matriz TF-IDF:", engine.tfidf_matrix.shape, "| entradas no cero:", engine.tfidf_matrix.nnz)
example_term = "aspirin" if "aspirin" in engine.vocabulary else sorted(engine.vocabulary)[0]
print(f"Posting list de '{example_term}' (muestra):", engine.inverted_index[example_term][:10])

## 4. Recuperación de una consulta externa

Se muestran rankings independientes y se validan formato, orden e IDs de documentos.
El buscador admite consultas nuevas, no solamente las incluidas en el dataset.

In [ ]:
external_query = "Aspirin reduces the risk of cardiovascular disease"
for model in ("tfidf", "bm25"):
    results = engine.search(external_query, model=model, top_k=5)
    assert len(results) <= 5
    assert len({r["document_id"] for r in results}) == len(results)
    assert all(r["document_id"] in doc_ids for r in results)
    assert [r["score"] for r in results] == sorted([r["score"] for r in results], reverse=True)
    print("\nMODELO:", model.upper())
    display(pd.DataFrame(results)[["rank", "document_id", "score", "title"]])

## 5. Selección exacta de consultas para evaluación

**No se elige un subconjunto favorable.** Se evalúan todas las consultas con qrels positivos
asignadas a `test`. La tabla permite identificar exactamente cuáles y cuántas se usaron.

In [ ]:
test_query_ids = sorted(qid for qid, rels in test_qrels.items() if any(value > 0 for value in rels.values()))
selected_queries = pd.DataFrame([
    {"query_id": qid, "query_text": queries[qid], "relevant_documents": sum(s > 0 for s in test_qrels[qid].values())}
    for qid in test_query_ids
])
print("Total de consultas evaluadas:", len(selected_queries))
display(selected_queries.head(10))
assert len(selected_queries) > 0

## 6. Evaluación reproducible: P@10, Recall@10 y NDCG@10

**Métricas macro:** primero se calcula la métrica para cada consulta de `test`
y después su media aritmética. P@10 siempre usa denominador 10, incluso si hay menos
relevantes. NDCG@10 tiene en cuenta el orden y los grados de relevancia.

In [ ]:
summary, per_query = evaluate_models(engine, queries, test_qrels)
metrics_df = pd.DataFrame(summary).set_index("model")
metrics_by_query_df = pd.DataFrame(per_query)
print("Resumen de métricas en TEST:")
display(metrics_df.style.format({
    "precision_at_10": "{:.4f}", "recall_at_10": "{:.4f}", "ndcg_at_10": "{:.4f}"
}))
print("Primeras métricas por consulta:")
display(metrics_by_query_df.head(8))
assert set(metrics_df.index) == {"bm25", "tfidf"}
assert metrics_df["queries_evaluated"].nunique() == 1

## 7. Comparación e interpretación

Se calcula **BM25 − TF-IDF** en cada métrica. El signo del diferencial se interpreta
sin presuponer cuál modelo gana. Debido a que SCIFACT suele tener pocos documentos
relevantes por consulta, P@10 puede ser baja incluso para rankings útiles.

In [ ]:
delta = metrics_df.loc["bm25", ["precision_at_10", "recall_at_10", "ndcg_at_10"]] - metrics_df.loc["tfidf", ["precision_at_10", "recall_at_10", "ndcg_at_10"]]
print("Diferencia BM25 - TF-IDF:")
display(delta.to_frame(name="delta"))
for metric, value in delta.items():
    winner = "BM25" if value > 0 else "TF-IDF" if value < 0 else "empate"
    print(f"{metric}: {winner} (Δ={value:+.4f})")
print("Interpretación: BM25 normaliza la frecuencia por longitud documental; "
      "TF-IDF compara vectores L2. Las diferencias observadas son descriptivas, no significancia estadística.")

## 8. Prueba del servicio web: contrato JSON

El examen exige dos endpoints comprobables desde Postman. Aquí se verifican
sin iniciar un servidor externo, ejecutando la **misma aplicación FastAPI**
con el corpus cargado y el índice real.

In [ ]:
from fastapi.testclient import TestClient
from app.main import app

with TestClient(app) as client:
    health = client.get("/health")
    print("GET /health:", health.status_code, health.json())
    assert health.status_code == 200
    for model in ("bm25", "tfidf"):
        response = client.post("/search", json={"query": external_query, "model": model, "top_k": 3})
        print(f"POST /search ({model}):", response.status_code, response.json())
        assert response.status_code == 200
        assert len(response.json()["results"]) <= 3
    bad = client.post("/search", json={"query": "", "model": "bm25"})
    print("Validación de consulta vacía:", bad.status_code)
    assert bad.status_code == 422

## 9. Guardado de evidencia

Se exportan las métricas globales y por consulta en `results/`. El **notebook
con outputs visibles**, guardado tras `Run All`, constituye el entregable principal,
junto a la URL pública del servicio.

In [ ]:
import json
OUT = ROOT / "results"
OUT.mkdir(exist_ok=True)
(OUT / "metrics_test.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
metrics_by_query_df.to_csv(OUT / "metrics_per_query_test.csv", index=False)
selected_queries.to_csv(OUT / "test_queries_evaluated.csv", index=False)
print("Guardado:", [p.name for p in OUT.glob("*") if p.is_file()])
print("Recuerda guardar el notebook con las salidas y comprobar la URL pública antes de entregar.")

## 10. Despliegue y endpoint público (evidencia de entrega)

1. Ejecutar `python -m scripts.prepare_data`, `python -m scripts.evaluate` y `pytest -q`.
2. Publicar en Render utilizando `render.yaml` (descarga corpus durante el build).
3. Probar la URL final con `python -m scripts.smoke_test https://<tu-servicio>.onrender.com`.
4. Entregar la URL comprobada y **este notebook con outputs**.

**Limitación de entrega:** la URL no se genera mediante la ejecución del notebook; es necesario
conectar el repositorio a un proveedor de hosting y verificar que el servicio esté disponible.
